In [ ]:
import tensorflow as tf
from tensorflow.keras.applications.inception_v3 import InceptionV3
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import Callback, EarlyStopping


# Calculate the macro F1 score
class MacroF1Score(tf.keras.metrics.Metric):
    def __init__(self, name='macro_f1_score', **kwargs):
        super(MacroF1Score, self).__init__(name=name, **kwargs)
        self.true_positives = self.add_weight(name='tp', initializer='zeros')
        self.false_positives = self.add_weight(name='fp', initializer='zeros')
        self.false_negatives = self.add_weight(name='fn', initializer='zeros')
        self.true_negatives = self.add_weight(name='tn', initializer='zeros')

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_pred = tf.round(y_pred)

        tp = tf.reduce_sum(tf.cast(y_true * y_pred, tf.float32))
        tn = tf.reduce_sum(tf.cast((1 - y_true) * (1 - y_pred), tf.float32))
        fp = tf.reduce_sum(tf.cast((1 - y_true) * y_pred, tf.float32))
        fn = tf.reduce_sum(tf.cast(y_true * (1 - y_pred), tf.float32))

        self.true_positives.assign_add(tp)
        self.false_positives.assign_add(fp)
        self.false_negatives.assign_add(fn)
        self.true_negatives.assign_add(tn)

    def result(self):
        precision_pos = self.true_positives / (self.true_positives + self.false_positives + tf.keras.backend.epsilon())
        recall_pos = self.true_positives / (self.true_positives + self.false_negatives + tf.keras.backend.epsilon())
        f1_pos = 2 * (precision_pos * recall_pos) / (precision_pos + recall_pos + tf.keras.backend.epsilon())

        precision_neg = self.true_negatives / (self.true_negatives + self.false_negatives + tf.keras.backend.epsilon())
        recall_neg = self.true_negatives / (self.true_negatives + self.false_positives + tf.keras.backend.epsilon())
        f1_neg = 2 * (precision_neg * recall_neg) / (precision_neg + recall_neg + tf.keras.backend.epsilon())

        # Macro F1 = average of F1 for class 0 and 1
        return (f1_pos + f1_neg) / 2

    def reset_states(self):
        self.true_positives.assign(0.0)
        self.false_positives.assign(0.0)
        self.false_negatives.assign(0.0)
        self.true_negatives.assign(0.0)

# Configuration
IMG_SIZE = (299, 299)
BATCH_SIZE = 32
TRAIN_DIR = "/kaggle/input/mmhs11k-urdu/MMHS11K_RGB_train/MMHS11K_RGB_train"
VAL_DIR = "/kaggle/input/mmhs11k-urdu/MMHS11K_RGB_test/MMHS11K_RGB_test"

# Data Generators
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=True,
    classes=['No_Hate', 'Hate']
)

val_generator = val_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False,
    classes=['No_Hate', 'Hate']
)

# Model Architecture
base_model = InceptionV3(weights='imagenet', include_top=False, input_shape=IMG_SIZE + (3,))
base_model.trainable = False
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation='relu')(x)
x = Dropout(0.4)(x)
predictions = Dense(1, activation='sigmoid')(x)
model = Model(inputs=base_model.input, outputs=predictions)

# Compile Model
model.compile(optimizer=Adam(learning_rate=0.0001),
              loss='binary_crossentropy',
              metrics=['accuracy', MacroF1Score()])

# Callback to save best model based only on TEST Macro F1
class BestTestMacroF1ModelSaver(Callback):
    def __init__(self, test_data, save_path='best_test_macro_f1_model.keras'):
        super(BestTestMacroF1ModelSaver, self).__init__()
        self.test_data = test_data
        self.best_test_macro_f1 = 0.0
        self.save_path = save_path

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        train_loss = logs.get('loss', 0.0)
        train_acc = logs.get('accuracy', 0.0)
        train_f1 = logs.get('macro_f1_score', 0.0)

        val_loss = logs.get('val_loss', 0.0)
        val_acc = logs.get('val_accuracy', 0.0)
        val_f1 = logs.get('val_macro_f1_score', 0.0)

        test_loss, test_acc, test_f1 = self.model.evaluate(self.test_data, verbose=0)

        # Print all metrics
        print(f"\nEpoch {epoch+1}")
        print(f"Train - Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, F1: {train_f1:.4f}")
        print(f"Val   - Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1: {val_f1:.4f}")
        print(f"Test  - Loss: {test_loss:.4f}, Acc: {test_acc:.4f}, F1: {test_f1:.4f}")

        if test_f1 > self.best_test_macro_f1:
            print(f"Saving model with improved TEST Macro F1: {test_f1:.4f}")
            self.best_test_macro_f1 = test_f1
            self.model.save(self.save_path)

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=5,
    callbacks=[
        EarlyStopping(patience=3, restore_best_weights=True),
        BestTestMacroF1ModelSaver(val_generator, save_path='best_test_macro_f1_model.keras')
    ]
)

base_model.trainable = True
model.compile(optimizer=Adam(learning_rate=0.00001),
              loss='binary_crossentropy',
              metrics=['accuracy', MacroF1Score()])

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    callbacks=[
        EarlyStopping(patience=3, restore_best_weights=True),
        BestTestMacroF1ModelSaver(val_generator, save_path='best_test_macro_f1_model.keras')
    ]
)

print("Training complete! Final model saved.")
